# Phase 3D — Apples-to-apples baseline and Numba validation

This is a **bridging experiment**, not a replacement for historical Phase 1 (512-feature TF-IDF) or Phase 2C (Gram matrix). All methods below search **the same real normalized MiniLM 384-d vectors**, use identical in-corpus query vectors and top-10 ranking, and run in the same Colab session.

**Methods:** Phase 1-style pure Python cosine (three explicit loops), NumPy sequential, NumPy batched BLAS, Numba serial, Numba `parallel=True` with `prange` over independent queries. Numba compilation is warmed up and excluded. Python baseline is intentionally restricted to smaller corpus/query counts; a large pure-Python search is not silently extrapolated. Numba is compared against NumPy for the **same top-k workload**. Earlier Phase 3B ThreadPool/ProcessPool results remain separate-session comparisons; this notebook does not re-run them.

**Important correctness:** normalized vectors allow cosine to equal dot product mathematically, but floating-point accumulation can differ; top-10 near ties may swap. We verify score closeness and top-10 **ID sets** and record ordered-match rates. Do not claim exactly identical ranking for ties.

In [ ]:
%pip -q install numba threadpoolctl pandas matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os,time,platform
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from numba import njit,prange,set_num_threads,get_num_threads
from threadpoolctl import threadpool_limits
ROOT=Path('/content/drive/MyDrive/HP-SS-Engine')
OUT=ROOT/'phase3_parallelism/benchmarks';FIG=ROOT/'phase3_parallelism/figures'
OUT.mkdir(parents=True,exist_ok=True);FIG.mkdir(parents=True,exist_ok=True)
X=np.load(ROOT/'phase2_matrix_ops/2A_sparse_dense/qwen_20k_minilm_384_float32.npy').astype(np.float32,copy=False)
assert X.shape==(20000,384)
X=np.ascontiguousarray(X/np.maximum(np.linalg.norm(X,axis=1,keepdims=True),1e-12))
print('CPU count',os.cpu_count(),'shape',X.shape,'Python',platform.python_version())

In [ ]:
def python_cosine(a,b):
    # Phase 1 algorithm structure: explicit dot + two norm loops.
    dot=0.0;aa=0.0;bb=0.0
    for j in range(len(a)): dot+=float(a[j])*float(b[j])
    for j in range(len(a)): aa+=float(a[j])*float(a[j])
    for j in range(len(b)): bb+=float(b[j])*float(b[j])
    return dot/(aa**0.5*bb**0.5) if aa>0 and bb>0 else 0.0

def rank_topk(scores,k=10):
    return np.lexsort((np.arange(len(scores)),-scores))[:k]
def pure_python_search(Q,C):
    return [rank_topk(np.array([python_cosine(q,x) for x in C],dtype=np.float64)) for q in Q]
def numpy_sequential_scores(Q,C):
    return np.stack([C@q for q in Q])
def numpy_batched_scores(Q,C):
    return Q@C.T

@njit
def numba_serial_scores(Q,C):
    n_queries,d=Q.shape;n_docs=C.shape[0]
    scores=np.empty((n_queries,n_docs),dtype=np.float32)
    for i in range(n_queries):
        for j in range(n_docs):
            s=np.float32(0.0)
            for t in range(d): s+=Q[i,t]*C[j,t]
            scores[i,j]=s
    return scores

@njit(parallel=True)
def numba_parallel_scores(Q,C):
    n_queries,d=Q.shape;n_docs=C.shape[0]
    scores=np.empty((n_queries,n_docs),dtype=np.float32)
    for i in prange(n_queries):
        for j in range(n_docs):
            s=np.float32(0.0)
            for t in range(d): s+=Q[i,t]*C[j,t]
            scores[i,j]=s
    return scores

def rank_rows(S):
    return [rank_topk(row) for row in S]

# Compile JIT with a tiny representative input OUTSIDE timing.
numba_serial_scores(X[:2],X[:4]);numba_parallel_scores(X[:2],X[:4]);print('Numba warmed up')

In [ ]:
PILOT=True
DOC_SIZES=[100,500] if PILOT else [100,500,1000,5000,10000,20000]
QUERY_COUNTS=[4,8] if PILOT else [4,16,64]
PYTHON_MAX_PAIRS=4000 if PILOT else 4000
REPEATS=3 if PILOT else 5
BLAS_THREADS=1
NUMBA_THREADS=min(2,os.cpu_count() or 2)
set_num_threads(NUMBA_THREADS)
SEED=20261020
rows=[];checks=[]
with threadpool_limits(limits=BLAS_THREADS,user_api='blas'):
 for n in DOC_SIZES:
  C=X[:n]
  for nq in QUERY_COUNTS:
   rng=np.random.default_rng(SEED+n+nq)
   Q=np.ascontiguousarray(C[rng.choice(n,size=nq,replace=nq>n)])
   ref=numpy_sequential_scores(Q,C)
   methods={'numpy_sequential':lambda:rank_rows(numpy_sequential_scores(Q,C)),
            'batched_blas':lambda:rank_rows(numpy_batched_scores(Q,C)),
            'numba_serial':lambda:rank_rows(numba_serial_scores(Q,C)),
            'numba_parallel':lambda:rank_rows(numba_parallel_scores(Q,C))}
   if n*nq<=PYTHON_MAX_PAIRS:
    methods['pure_python_cosine']=lambda:pure_python_search(Q,C)
   ref_top=rank_rows(ref)
   # Score validation for NumPy and Numba. Dot and cosine differ only by float rounding after normalization.
   for label,score_fn in [('batched_blas',numpy_batched_scores),('numba_serial',numba_serial_scores),('numba_parallel',numba_parallel_scores)]:
    S=score_fn(Q,C)
    max_err=float(np.max(np.abs(S-ref)))
    assert np.allclose(S,ref,rtol=2e-4,atol=2e-5),(label,n,nq,max_err)
    ids=rank_rows(S)
    set_match=np.mean([set(a)==set(b) for a,b in zip(ref_top,ids)])
    order_match=np.mean([np.array_equal(a,b) for a,b in zip(ref_top,ids)])
    checks.append(dict(n_docs=n,n_queries=nq,method=label,max_abs_score_error=max_err,top10_set_match_rate=set_match,top10_ordered_match_rate=order_match))
    assert set_match==1.0,(label,n,nq,'Top10 sets differed; investigate near-ties before proceeding')
   if 'pure_python_cosine' in methods:
    py_ids=methods['pure_python_cosine']()
    py_set_match=np.mean([set(a)==set(b) for a,b in zip(ref_top,py_ids)])
    py_order_match=np.mean([np.array_equal(a,b) for a,b in zip(ref_top,py_ids)])
    checks.append(dict(n_docs=n,n_queries=nq,method='pure_python_cosine',max_abs_score_error=np.nan,top10_set_match_rate=py_set_match,top10_ordered_match_rate=py_order_match))
    assert py_set_match==1.0,(n,nq,'Python top10 sets differ; inspect scores/ties')
   for method,fn in methods.items():
    fn() # warm the execution path
    for rep in range(REPEATS):
     t0=time.perf_counter_ns();result=fn();elapsed_ms=(time.perf_counter_ns()-t0)/1e6
     rows.append(dict(mode='pilot' if PILOT else 'full',method=method,n_docs=n,n_queries=nq,blas_threads=BLAS_THREADS,numba_threads=NUMBA_THREADS,repeat=rep,elapsed_ms=elapsed_ms))
   print('done',n,nq,'methods',list(methods))
tag='pilot' if PILOT else 'full'
raw=pd.DataFrame(rows);correct=pd.DataFrame(checks)
raw.to_csv(OUT/f'phase3d_same_vectors_raw_{tag}.csv',index=False)
correct.to_csv(OUT/f'phase3d_same_vectors_correctness_{tag}.csv',index=False)
summary=raw.groupby(['method','n_docs','n_queries','blas_threads','numba_threads'],as_index=False).agg(median_ms=('elapsed_ms','median'),mean_ms=('elapsed_ms','mean'),std_ms=('elapsed_ms','std'))
seq=summary[summary.method=='numpy_sequential'][['n_docs','n_queries','median_ms']].rename(columns={'median_ms':'numpy_baseline_ms'})
summary=summary.merge(seq,on=['n_docs','n_queries'],how='left')
summary['speedup_vs_numpy']=summary.numpy_baseline_ms/summary.median_ms
summary.to_csv(OUT/f'phase3d_same_vectors_summary_{tag}.csv',index=False)
print('Measurements',len(raw),'configurations',len(summary))
print(summary.to_string(index=False))
print('Correctness checks:');print(correct.to_string(index=False))

In [ ]:
focus=summary[(summary.n_docs==max(DOC_SIZES))&(summary.n_queries==max(QUERY_COUNTS))].sort_values('median_ms')
fig,ax=plt.subplots(figsize=(9,4.5))
ax.bar(focus.method,focus.median_ms)
ax.set(ylabel='Median search+ranking time (ms)',title=f'Phase 3D same-vector comparison: {max(DOC_SIZES)} docs × {max(QUERY_COUNTS)} queries')
ax.tick_params(axis='x',rotation=25)
fig.tight_layout();p=FIG/f'phase3d_same_vectors_{tag}.png';fig.savefig(p,dpi=160);plt.show();print('Saved',p)
print('Python baseline is measured only where n_docs*n_queries <=',PYTHON_MAX_PAIRS)

## Interpretation and final-report integration

- Every method in this notebook uses the **same MiniLM vector space**, document IDs, queries and top-10 task. This addresses the Phase 1-to-Phase 3 representation mismatch **without overwriting historical Phase 1 results**.
- `pure_python_cosine` is deliberately naive, like Phase 1; normalization is recomputed per document. The NumPy/Numba paths use cached normalized vectors and dot products. Thus differences reflect **both** arithmetic implementation and pre-normalization reuse; label this clearly. For a stricter implementation-only test, add a pure Python dot-only baseline on the normalized vectors.
- Search timing includes scoring + top-10 ranking. Normalization and loading are excluded. Numba JIT compilation is excluded. Pure Python large workloads are skipped by design, not estimated.
- `numba_parallel` uses `prange` across independent queries; when query count is small, limited parallelism and scheduling overhead may dominate.
- Phase 3A/B ThreadPool and ProcessPool still have their own matched baselines. Cross-session absolute times cannot be combined into a single benchmark ranking.
- Phase 3C asyncio simulated I/O remains a separate type of workload. No numerical speedup comparison with CPU similarity search is justified.
- **Next:** inspect pilot output before switching `PILOT=False`. Update Phase 3 final report using *actual measured* Phase 3D values.